# Stage 4 — Modeling

Goal: score each customer by how much they look like customers who churned.

Two rules:
1. **Don't touch the test set.** All comparisons use 5-fold cross-validation on the training set.
2. **Split first.** Nothing (scaler, encoder, bins) is fitted before the split.

**Limit:** the data has no dates, so this is risk scoring, not a forecast of next month.

## Setup

In [9]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from features import load_modelling_data, split, make_preprocessor, NUMERIC_FEATURES
from modeling import ContractRule, score_model, CV, SCORING

pd.set_option("display.width", 200)

data = load_modelling_data()
X_train, X_test, y_train, y_test = split(data)

7,032 customers for modelling (11 with tenure = 0 excluded)
train 5,274 rows, churn 0.266
test  1,758 rows, churn 0.266   (untouched until Stage 5)


We remove the 11 tenure-0 customers **before** splitting. They can't have churned yet, so their label isn't real.

## Baselines

Two simple baselines to beat:

In [10]:
baselines = [
    score_model(DummyClassifier(strategy="most_frequent"), X_train, y_train, "Predict nobody churns", SCORING),
    score_model(ContractRule(), X_train, y_train, "Rule: month-to-month", SCORING),
]
pd.DataFrame(baselines).set_index("model")[["accuracy", "precision", "recall", "f1"]].round(3)

,accuracy,precision,recall,f1
model,,,,
Predict nobody churns,0.734,0.000,0.000,0.000
Rule: month-to-month,0.655,0.428,0.885,0.577


**The more "accurate" baseline is useless.** "Nobody churns" gets 73.4% accuracy but flags no one. The month-to-month rule has lower accuracy but catches **88.5%** of churners.

So the bar to beat is **F1 = 0.577**, not accuracy.

The rule's precision is only 0.428: 57% of the people it flags wouldn't have churned. A model should cut that waste.

## Feature options

Four open questions from Stages 2–3. Cross-validation decides.

In [11]:
def logreg(numeric, bin_tenure=False, **kwargs):
    return Pipeline([("prep", make_preprocessor("linear", numeric, bin_tenure)),
                     ("model", LogisticRegression(max_iter=2000, **kwargs))])

variants = [
    ("A. all 3 numerics (naive)",  logreg(NUMERIC_FEATURES)),
    ("B. drop MonthlyCharges",     logreg(["tenure", "TotalCharges"])),
    ("C. drop Monthly + Total",    logreg(["tenure"])),
    ("D. C + tenure binned",       logreg(["tenure"], bin_tenure=True)),
]
results = pd.DataFrame([score_model(m, X_train, y_train, n) for n, m in variants]).set_index("model")
results[["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "roc_auc_std"]].round(3)

,accuracy,precision,recall,f1,roc_auc,pr_auc,roc_auc_std
model,,,,,,,
A. all 3 numerics (naive),0.801,0.650,0.544,0.592,0.846,0.662,0.020
B. drop MonthlyCharges,0.800,0.649,0.543,0.591,0.846,0.662,0.020
C. drop Monthly + Total,0.800,0.648,0.538,0.588,0.844,0.660,0.020
D. C + tenure binned,0.806,0.666,0.540,0.596,0.848,0.666,0.022


**All four options perform the same.** ROC-AUC ranges 0.844–0.848, but fold-to-fold noise is **0.020**, five times bigger. The differences aren't real.

So we pick the simplest. **Chosen: option C** (21 features, least overlap, same score).

- Dropping `MonthlyCharges` (VIF **878**, it's just the sum of the services) cost only 0.002.
- `TotalCharges` adds nothing.
- Binning tenure (option D) is 0.004 better but adds 5 features and is noisier.

Tenure bins are learned *inside* the pipeline, so each fold uses only its own training data. Binning the full data first would leak test info.

## Class imbalance

Only 26.6% of customers churn. Common fixes are resampling or class weights. We skip both. Here's why:

In [12]:
weighted = [score_model(logreg(["tenure"]), X_train, y_train, "C, natural distribution"),
            score_model(logreg(["tenure"], class_weight="balanced"), X_train, y_train, "C, class_weight=balanced")]
print(pd.DataFrame(weighted).set_index("model")[["precision", "recall", "f1", "roc_auc", "pr_auc"]].round(3).to_string())

# Is it a better model, or the same model at a different threshold?
p_natural = cross_val_predict(logreg(["tenure"]), X_train, y_train, cv=CV, method="predict_proba")[:, 1]
p_balanced = cross_val_predict(logreg(["tenure"], class_weight="balanced"), X_train, y_train,
                               cv=CV, method="predict_proba")[:, 1]

print(f"\nSpearman correlation of the two rankings : "
      f"{pd.Series(p_natural).corr(pd.Series(p_balanced), method='spearman'):.4f}")
print(f"mean predicted probability   natural {p_natural.mean():.3f} | balanced {p_balanced.mean():.3f}")
print(f"actual churn rate                     {y_train.mean():.3f}")

                          precision  recall     f1  roc_auc  pr_auc
model                                                              
C, natural distribution       0.648   0.538  0.588    0.844   0.660
C, class_weight=balanced      0.522   0.795  0.630    0.844   0.659

Spearman correlation of the two rankings : 0.9995
mean predicted probability   natural 0.266 | balanced 0.414
actual churn rate                     0.266


F1 goes up (0.588 → 0.630), but **ROC-AUC and PR-AUC don't change, and the rankings match at 0.9995.** It's the same model with a different threshold.

And it has a cost: normal training predicts **0.266** on average (the true rate). Balanced weights push it to **0.414**.

Stage 6 multiplies probability by customer value, so the probabilities must be honest. We keep them accurate and pick a threshold in Stage 5 instead.

Same reason we skip SMOTE: it creates fake churners and distorts the rate.

## Gradient boosting

Can a more flexible model find patterns the simple one misses?

In [13]:
def boosted(numeric, **kwargs):
    # No scaling: tree splits are scale-invariant, so standardising changes nothing.
    return Pipeline([("prep", make_preprocessor("tree", numeric)),
                     ("model", HistGradientBoostingClassifier(random_state=42, **kwargs))])

# HistGradientBoosting only enables early stopping above 10,000 rows. With 5,274 the
# default runs all 100 iterations unchecked, so the first row below is overfitting.
trees = [
    ("GB default",                boosted(["tenure"])),
    ("GB + early stopping",       boosted(["tenure"], early_stopping=True, validation_fraction=0.15)),
    ("GB shallow (4 leaves)",     boosted(["tenure"], max_leaf_nodes=4, learning_rate=0.05, max_iter=300)),
    ("GB shallow + all numerics", boosted(NUMERIC_FEATURES, max_leaf_nodes=4, learning_rate=0.05, max_iter=300)),
]
tree_results = pd.DataFrame([score_model(m, X_train, y_train, n) for n, m in trees]).set_index("model")
print(tree_results[["precision", "recall", "f1", "roc_auc", "pr_auc"]].round(3).to_string())
print("\nLR variant C for reference:   roc_auc 0.844   pr_auc 0.660   f1 0.588")

                           precision  recall     f1  roc_auc  pr_auc
model                                                               
GB default                     0.638   0.526  0.576    0.839   0.651
GB + early stopping            0.651   0.511  0.572    0.844   0.657
GB shallow (4 leaves)          0.659   0.529  0.587    0.850   0.670
GB shallow + all numerics      0.662   0.537  0.592    0.848   0.667

LR variant C for reference:   roc_auc 0.844   pr_auc 0.660   f1 0.588


Two notes:

- **The default model overfit.** Limiting it to 4 leaves improved ROC-AUC by 0.011.
- **The extra numeric columns don't help the tree either** (0.848 vs 0.850, just noise).

The tree scores 0.850 vs 0.844 for logistic regression. With noise of 0.020, we need a fold-by-fold comparison.

In [6]:
lr_best = logreg(["tenure"])
gb_best = boosted(["tenure"], max_leaf_nodes=4, learning_rate=0.05, max_iter=300)

# Same folds for both models, so differences are paired rather than two noisy averages.
scored = {name: cross_validate(m, X_train, y_train, cv=CV, scoring=["roc_auc", "average_precision"])
          for name, m in [("LR", lr_best), ("GB", gb_best)]}

for metric, key in [("ROC-AUC", "test_roc_auc"), ("PR-AUC", "test_average_precision")]:
    difference = scored["GB"][key] - scored["LR"][key]
    print(f"{metric}: GB - LR per fold {np.round(difference, 4)}")
    print(f"   mean {difference.mean():+.4f}, GB wins {int((difference > 0).sum())}/5 folds\n")

ROC-AUC: GB - LR per fold [0.0005 0.0007 0.0042 0.0152 0.0076]
   mean +0.0056, GB wins 5/5 folds

PR-AUC: GB - LR per fold [-0.0018 -0.0075  0.0158  0.0239  0.0214]
   mean +0.0104, GB wins 3/5 folds



- **ROC-AUC:** the tree wins every fold, but only by 0.006.
- **PR-AUC:** the tree loses 2 of 5 folds.

PR-AUC matters more here because it focuses on churners, which is what a contact list is made of.

### Decision: logistic regression

The tree is slightly better at ranking, but **+0.006 isn't worth losing explainability.** Stage 6 needs to explain why each customer was flagged.

The tie is a finding too: the patterns in this data are mostly simple, so a flexible model has little extra to find.

## Are the probabilities accurate?

Stage 6 multiplies probability by customer value, so a score of 0.6 should mean about 60% churn.

**Careful:** on training data, logistic regression's average prediction always equals the true rate. That proves nothing. We need to check on data the model didn't see.

In [7]:
from sklearn.metrics import brier_score_loss

# Out-of-fold: every row scored by a model that never saw it.
p_oof = cross_val_predict(logreg(["tenure"]), X_train, y_train, cv=CV, method="predict_proba")[:, 1]

deciles = pd.DataFrame({"predicted": p_oof, "actual": y_train.to_numpy(), "bin": pd.qcut(p_oof, 10)})
calibration = deciles.groupby("bin", observed=True).agg(
    customers=("actual", "size"), predicted=("predicted", "mean"), observed=("actual", "mean"))
calibration["gap_pp"] = ((calibration["observed"] - calibration["predicted"]) * 100).round(1)
print(calibration.round(3).to_string())

base_rate = np.full(len(y_train), y_train.mean())
print(f"\nBrier {brier_score_loss(y_train, p_oof):.4f}"
      f"  (always predicting the base rate scores {brier_score_loss(y_train, base_rate):.4f})")
print(f"worst decile gap {calibration['gap_pp'].abs().max():.1f}pp")

                   customers  predicted  observed  gap_pp
bin                                                      
(0.00215, 0.0151]        528      0.010     0.013     0.4
(0.0151, 0.0338]         527      0.024     0.025     0.1
(0.0338, 0.0644]         527      0.047     0.057     1.0
(0.0644, 0.115]          528      0.087     0.085    -0.2
(0.115, 0.18]            527      0.148     0.152     0.4
(0.18, 0.267]            527      0.221     0.226     0.4
(0.267, 0.387]           528      0.324     0.347     2.3
(0.387, 0.536]           527      0.456     0.412    -4.4
(0.536, 0.669]           527      0.601     0.583    -1.9
(0.669, 0.889]           528      0.741     0.759     1.8

Brier 0.1359  (always predicting the base rate scores 0.1952)
worst decile gap 4.4pp


Predicted and actual match well in all ten groups: worst gap **4.4pp**, most under 2pp. The Brier score beats just guessing the average. The probabilities are usable.

## Fit and save

In [8]:
from train_model import build_model, MODEL_PATH

model = build_model().fit(X_train, y_train)
print(f"{model['prep'].transform(X_train).shape[1]} encoded features")
print(f"saved by src/train_model.py to {MODEL_PATH.name}")

21 encoded features
saved by src/train_model.py to churn_logistic_regression.joblib


---
# Summary

| Decision | Result | Why |
|---|---|---|
| Customers | 7,032 (11 tenure-0 removed) | Their label isn't real |
| Split | 75/25, before any fitting | Avoid leakage |
| `MonthlyCharges` | **Dropped** | Sum of services; costs 0.002 |
| `TotalCharges` | **Dropped** | Adds nothing |
| `tenure` | **One number, not binned** | Bins add 5 features for +0.004 |
| Imbalance | **No reweighting** | Only moves the threshold and skews probabilities |
| Model | **Logistic regression**, 21 features | Tree only +0.006, inconsistent on PR-AUC |

Not done yet: test-set scoring, threshold, coefficients. That's Stage 5.

### For Stage 5

- **0.5 is the wrong threshold.** At 0.5 the model catches 53.8% of churners, *fewer* than the simple rule (88.5%). The model ranks better; it just needs a better cut-off.
- Probabilities are accurate on unseen folds: Brier 0.136 vs 0.195 baseline, worst gap 4.4pp.
- `Contract` has the biggest coefficient and may hide weaker signals.